<a href="https://colab.research.google.com/github/minju0236/Hankyung-Bootcamp/blob/main/Day1_a_(260330)_%ED%8A%B8%EB%A6%AC%26%ED%95%B4%EC%8B%9C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
%%writefile matrix.js

/**
 * 인접 행렬 (Adjacency Matrix)
 * - 2차원 배열을 사용하여 연결 상태를 0과 1로 저장
 */
class AdjacencyMatrix {
  constructor(vertices) {
    this.vertices = vertices; // 정점 이름 배열 (예: ['A', 'B', ...])
    this.size = vertices.length;
    // 초기 상태: 모두 0으로 채워진 size x size 행렬
    this.matrix = Array.from({ length: this.size }, () => Array(this.size).fill(0));
    // 정점 이름을 인덱스로 변환하기 위한 맵 (A -> 0, B -> 1)
    this.indexMap = {};
    vertices.forEach((v, i) => this.indexMap[v] = i);
  }

  // 방향성 간선 추가: v1 -> v2
  addEdge(v1, v2) {
    const i = this.indexMap[v1];
    const j = this.indexMap[v2];
    if (i !== undefined && j !== undefined) {
      this.matrix[i][j] = 1;
    }
  }

  printMatrix() {
    console.log("   " + this.vertices.join(" "));
    this.matrix.forEach((row, i) => {
      console.log(`${this.vertices[i]} [ ${row.join(" ")} ]`);
    });
  }
}

// --- 호출부 ---
const vList = ['A', 'B', 'C', 'D', 'E'];
const matrixGraph = new AdjacencyMatrix(vList);

matrixGraph.addEdge('A', 'B');
matrixGraph.addEdge('A', 'C');
matrixGraph.addEdge('D', 'B');
matrixGraph.addEdge('D', 'C');
matrixGraph.addEdge('D', 'E');
matrixGraph.addEdge('B', 'E');

console.log("=== 인접 행렬 결과 ===");
matrixGraph.printMatrix();

Writing matrix.js


In [2]:
!node matrix.js

=== 인접 행렬 결과 ===
   A B C D E
A [ 0 1 1 0 0 ]
B [ 0 0 0 0 1 ]
C [ 0 0 0 0 0 ]
D [ 0 1 1 0 1 ]
E [ 0 0 0 0 0 ]


In [3]:
%%writefile adjacency.js

/**
 * 인접 리스트 (Adjacency List)
 * - 각 정점을 Key로, 연결된 정점 배열을 Value로 갖는 객체(Map) 사용
 */
class AdjacencyList {
  constructor() {
    this.list = {};
  }

  addVertex(v) {
    if (!this.list[v]) this.list[v] = [];
  }

  // 방향성 간선 추가: v1 -> v2
  addEdge(v1, v2) {
    this.addVertex(v1);
    this.addVertex(v2);
    this.list[v1].push(v2);
  }

  printList() {
    for (let v in this.list) {
      console.log(`${v} => [ ${this.list[v].join(", ")} ]`);
    }
  }
}

// --- 호출부 ---
const listGraph = new AdjacencyList();

// 이미지 데이터 기반 간선 추가
const directedEdges = [
  ['A', 'B'], ['A', 'C'],
  ['D', 'B'], ['D', 'C'], ['D', 'E'],
  ['B', 'E']
];

directedEdges.forEach(([from, to]) => listGraph.addEdge(from, to));

console.log("\n=== 인접 리스트 결과 ===");
listGraph.printList();

Writing adjacency.js


In [4]:
!node adjacency.js


=== 인접 리스트 결과 ===
A => [ B, C ]
B => [ E ]
C => [  ]
D => [ B, C, E ]
E => [  ]


In [5]:
%%writefile graph.js

class Graph {
  constructor() {
    this.adjacencyList = {}; // 정점별 인접 노드를 배열로 관리하는 인접 리스트
  }

  // 정점 추가: 리스트에 해당 키가 없으면 빈 배열로 초기화
  addVertex(v) {
    if (!this.adjacencyList[v]) this.adjacencyList[v] = [];
  }

  // 간선 추가 (방향성: src -> dest)
  addEdge(src, dest) {
    this.addVertex(src);
    this.addVertex(dest);
    this.adjacencyList[src].push(dest);
  }

  /**
   * DFS (깊이 우선 탐색): 재귀 방식
   * @param {string} start - 탐색을 시작할 정점 이름
   */
  dfs(start) {
    const result = [];      // 방문한 순서대로 저장할 배열
    const visited = {};     // 중복 방문을 방지하기 위한 체크 객체 (Hash Map 역할)
    const list = this.adjacencyList; // 인접 리스트 참조

    // 재귀적으로 동작하는 내부 헬퍼 함수
    (function traverse(vertex) {
      if (!vertex) return;       // 정점이 없으면 종료
      visited[vertex] = true;    // 1. 현재 정점 방문 처리 (체크)
      result.push(vertex);       // 2. 결과 배열에 기록

      // 3. 현재 정점과 연결된 인접 노드들을 하나씩 확인
      list[vertex].forEach(neighbor => {
        // 아직 방문하지 않은 이웃 노드가 있다면 바로 그 노드로 깊게 침투(재귀 호출)
        if (!visited[neighbor]) {
          traverse(neighbor);
        }
      });
    })(start);

    return result;
  }

  /**
   * BFS (너비 우선 탐색): 큐 방식
   * @param {string} start - 탐색을 시작할 정점 이름
   */
  bfs(start) {
    const queue = [start];  // 방문 예정인 노드를 담는 큐 (FIFO)
    const result = [];      // 방문 완료 순서
    const visited = {};     // 방문 체크 객체

    visited[start] = true;  // 시작 노드 예약 처리

    // 큐가 빌 때까지 반복 (더 이상 갈 곳이 없을 때까지)
    while (queue.length > 0) {
      // 1. 큐의 맨 앞(가장 먼저 들어온 노드)을 꺼냄
      const current = queue.shift();
      result.push(current);

      // 2. 꺼낸 노드와 인접한 노드들을 수평적으로 확인
      this.adjacencyList[current].forEach(neighbor => {
        // 아직 방문하지 않은 이웃 노드라면
        if (!visited[neighbor]) {
          visited[neighbor] = true; // 방문 예정이라고 체크 (중복 방지)
          queue.push(neighbor);     // 큐의 뒤쪽에 추가하여 다음 순번으로 대기시킴
        }
      });
    }

    return result;
  }
}

/**
 * 실행 및 호출부
 */
const graph = new Graph();

// 이미지 속 Directed Graph 관계도 입력
const directedEdges = [
  ['A', 'B'], ['A', 'C'],
  ['D', 'B'], ['D', 'C'], ['D', 'E'],
  ['B', 'E']
];
directedEdges.forEach(([from, to]) => graph.addEdge(from, to));

console.log("--- 그래프 탐색 결과 ---");

// A에서 시작할 경우
console.log("DFS (시작: A) ->", graph.dfs('A'));
// 예상 결과: [ 'A', 'B', 'E', 'C' ] (A->B 방문 후 끝까지(E) 갔다가 돌아와서 C 방문)

console.log("BFS (시작: A) ->", graph.bfs('A'));
// 예상 결과: [ 'A', 'B', 'C', 'E' ] (A의 이웃 B, C를 먼저 다 보고 그다음 멀리 있는 E 방문)

// D에서 시작할 경우 (모든 정점 방문 가능)
console.log("BFS (시작: D) ->", graph.bfs('D'));
// 예상 결과: [ 'D', 'B', 'C', 'E' ]

Writing graph.js


In [6]:
!node graph.js

--- 그래프 탐색 결과 ---
DFS (시작: A) -> [ 'A', 'B', 'E', 'C' ]
BFS (시작: A) -> [ 'A', 'B', 'C', 'E' ]
BFS (시작: D) -> [ 'D', 'B', 'C', 'E' ]


In [7]:
%%writefile user.js

/**
 * 방향 그래프 (Directed Graph) 클래스 정의
 */
class DirectedGraph {
  constructor() {
    // 각 정점과 그에 연결된 인접 정점들을 저장할 객체
    this.adjacencyList = {};
  }

  // 정점(Vertex) 추가
  addVertex(vertex) {
    if (!this.adjacencyList[vertex]) {
      this.adjacencyList[vertex] = [];
      console.log(`[시스템] 정점 '${vertex}'가 생성되었습니다.`);
    }
  }

  // 방향성 간선(Edge) 추가 (v1 -> v2)
  addEdge(source, destination) {
    if (this.adjacencyList[source] && this.adjacencyList[destination]) {
      this.adjacencyList[source].push(destination);
      console.log(`[연결] ${source} → ${destination} (단방향)`);
    } else {
      console.error("[오류] 존재하지 않는 정점이 포함되어 있습니다.");
    }
  }

  // 그래프 전체 구조 출력
  display() {
    console.log("\n--- 현재 소셜 네트워크 관계도 ---");
    for (let user in this.adjacencyList) {
      const follows = this.adjacencyList[user].length > 0
                      ? this.adjacencyList[user].join(", ")
                      : "없음";
      console.log(`${user} 팔로우 목록: [ ${follows} ]`);
    }
  }
}

/**
 * 실행 및 호출부
 */
const sns = new DirectedGraph();

// 1. 사용자(정점) 등록
sns.addVertex("BTS_Official");
sns.addVertex("User_A");
sns.addVertex("User_B");

// 2. 관계(방향 간선) 설정
// 일반 유저들이 공식 계정을 팔로우함 (일방적 관계)
sns.addEdge("User_A", "BTS_Official");
sns.addEdge("User_B", "BTS_Official");

// 유저들끼리의 관계
sns.addEdge("User_A", "User_B"); // A가 B를 팔로우함

// 3. 네트워크 시각화 출력
sns.display();

/* [출력 결과]
BTS_Official 팔로우 목록: [ 없음 ] (영향력 있는 노드/Sink Node)
User_A 팔로우 목록: [ BTS_Official, User_B ]
User_B 팔로우 목록: [ BTS_Official ]
*/

Writing user.js


In [8]:
!node user.js

[시스템] 정점 'BTS_Official'가 생성되었습니다.
[시스템] 정점 'User_A'가 생성되었습니다.
[시스템] 정점 'User_B'가 생성되었습니다.
[연결] User_A → BTS_Official (단방향)
[연결] User_B → BTS_Official (단방향)
[연결] User_A → User_B (단방향)

--- 현재 소셜 네트워크 관계도 ---
BTS_Official 팔로우 목록: [ 없음 ]
User_A 팔로우 목록: [ BTS_Official, User_B ]
User_B 팔로우 목록: [ BTS_Official ]


In [11]:
%%writefile friend.js

/**
 * SNS 친구 추천 시스템
 */
class SocialNetwork {
  constructor() {
    this.adjList = {};
  }

  // 사용자(정점) 추가
  addUser(name) {
    if (!this.adjList[name]) this.adjList[name] = [];
  }

  // 친구 관계(무방향 간선) 추가
  addFriendship(u1, u2) {
    this.adjList[u1].push(u2);
    this.adjList[u2].push(u1);
  }

  // 2촌 관계(친구의 친구) 추천 로직
  recommend(target) {
    if (!this.adjList[target]) return [];

    const myFriends = new Set(this.adjList[target]); // 나의 1촌 친구들
    const suggestions = new Set();

    // 내 친구들의 친구 목록을 전수 조사
    for (let friend of myFriends) {
      for (let fof of this.adjList[friend]) {
        // 본인이 아니고, 이미 내 친구가 아닌 사람만 골라냄
        if (fof !== target && !myFriends.has(fof)) {
          suggestions.add(fof);
        }
      }
    }
    return Array.from(suggestions);
  }
}

// --- 호출 및 실행 ---
const sns = new SocialNetwork();
["철수", "영희", "민수", "지호", "혜진"].forEach(name => sns.addUser(name));

sns.addFriendship("철수", "영희"); // 철수-영희 친구
sns.addFriendship("철수", "민수"); // 철수-민수 친구
sns.addFriendship("영희", "지호"); // 영희-지호 친구 (철수의 2촌)
sns.addFriendship("민수", "혜진"); // 민수-혜진 친구 (철수의 2촌)

console.log(`[인맥 추천] 철수님과 알 수도 있는 사람:`, sns.recommend("철수"));
// 출력 결과: [ '지호', '혜진' ]

Overwriting friend.js


In [12]:
!node friend.js

[인맥 추천] 철수님과 알 수도 있는 사람: [ '지호', '혜진' ]


In [13]:
%%writefile delivery.js

/**
 * 배달 경로 소요 시간 관리
 */
class DeliveryMap {
  constructor() {
    this.locations = {};
  }

  addLocation(loc) {
    this.locations[loc] = [];
  }

  // 구간 정보 추가 (출발, 도착, 소요시간)
  addRoute(from, to, time) {
    this.locations[from].push({ target: to, cost: time });
    this.locations[to].push({ target: from, cost: time });
  }

  // 특정 위치의 배달 가능 경로 확인
  showRoutes(currentLoc) {
    console.log(`\n--- [${currentLoc}] 출발 배달 노선 ---`);
    this.locations[currentLoc].forEach(route => {
      console.log(`목적지: ${route.target} | 예상소요: ${route.cost}분`);
    });
  }
}

// --- 호출 및 실행 ---
const delivery = new DeliveryMap();
["강남", "역삼", "선릉", "논현"].forEach(loc => delivery.addLocation(loc));

delivery.addRoute("강남", "역삼", 10);
delivery.addRoute("역삼", "선릉", 15);
delivery.addRoute("강남", "논현", 8);

delivery.showRoutes("강남");
/* 출력 결과:
목적지: 역삼 | 예상소요: 10분
목적지: 논현 | 예상소요: 8분
*/

Writing delivery.js


In [14]:
!node delivery.js


--- [강남] 출발 배달 노선 ---
목적지: 역삼 | 예상소요: 10분
목적지: 논현 | 예상소요: 8분


In [15]:
%%writefile programing.js

/**
 * 수강 신청 선수과목 로드맵
 */
class CourseRoadmap {
  constructor() {
    this.courses = {};
  }

  addCourse(title) {
    this.courses[title] = [];
  }

  // 선수과목 관계 추가 (A를 들어야 B 수강 가능)
  addPrerequisite(pre, post) {
    this.courses[pre].push(post);
  }

  // 특정 과목 이후의 추천 학습 경로 (DFS 탐색)
  getLearningPath(startCourse) {
    const path = [];
    const visited = new Set();

    const traverse = (course) => {
      visited.add(course);
      path.push(course);
      this.courses[course].forEach(next => {
        if (!visited.has(next)) traverse(next);
      });
    };

    traverse(startCourse);
    return path.join(" → ");
  }
}

// --- 호출 및 실행 ---
const myLecture = new CourseRoadmap();
["프로그래밍 입문", "JS 기초", "리액트", "자료구조"].forEach(c => myLecture.addCourse(c));

myLecture.addPrerequisite("프로그래밍 입문", "JS 기초");
myLecture.addPrerequisite("JS 기초", "리액트");
myLecture.addPrerequisite("JS 기초", "자료구조");

console.log("\n[학습 로드맵]");
console.log("추천 경로:", myLecture.getLearningPath("프로그래밍 입문"));
// 출력 결과: 프로그래밍 입문 → JS 기초 → 리액트 → 자료구조

Writing programing.js


In [16]:
!node programing.js


[학습 로드맵]
추천 경로: 프로그래밍 입문 → JS 기초 → 리액트 → 자료구조


In [17]:
%%writefile start.js

console.log("시작"); // 가장 먼저 동기 코드인 콘솔로그가 실행된다 (call stack)

setTimeout(() => {
    console.log("타이머 콜백"); // setTimeout은 매크로태스크 큐로. 마이크로테스크가 다 비워진 후에 실행된다.
}, 0);

Promise.resolve().then(() => {
    console.log("프로미스 콜백"); // Promise.then은 마이크로태스크 큐로, 일반 태스크보다 먼저 실행된다.
});

console.log("끝"); // 그 다음 동기 코드가 실행된다 (call stack)

/* 결과:
시작 -> 끝 -> 프로미스 콜백 -> 타이머 콜백
*/

Writing start.js


In [18]:
!node start.js

시작
끝
프로미스 콜백
타이머 콜백


In [19]:
%%writefile sky.js

function longTask() {
    console.log("강하늘님의 요청 처리 중...");
    setTimeout(() => {
        console.log("1초 뒤 실행될 작업");
    }, 1000);
}

longTask();
// 1. longTask가 콜 스택에 들어감.
// 2. setTimeout이 실행되며 Web API에게 타이머를 맡김.
// 3. longTask 종료 후 콜 스택 비워짐.
// 4. 1000ms 후 Web API가 콜백을 Task Queue로 보냄.
// 5. 이벤트 루프가 스택이 비었음을 확인하고 큐에서 콜백을 스택으로 올림.

Writing sky.js


In [20]:
!node sky.js

강하늘님의 요청 처리 중...
1초 뒤 실행될 작업


In [21]:
%%writefile predict.js

setTimeout(() => console.log("A"), 0);

Promise.resolve().then(() => {
    console.log("B");
    return Promise.resolve();
}).then(() => console.log("C"));

console.log("D");

Writing predict.js


In [22]:
!node predict.js

D
B
C
A


In [23]:
%%writefile check.js

const checkUserAuth = new Promise((resolve, reject) => {
    const isAuthorized = true; // 김철수 사용자의 인증 여부 예시

    if (isAuthorized) {
        // Pending -> Fulfilled 상태 전이
        resolve("인증 성공");
    } else {
        // Pending -> Rejected 상태 전이
        reject(new Error("인증 실패"));
    }
});

checkUserAuth
    .then(result => console.log(result)) // Fulfilled 시 실행
    .catch(error => console.error(error)); // Rejected 시 실행

Writing check.js


In [24]:
!node check.js

인증 성공


In [25]:
%%writefile test.js

console.log("1. 동기 코드 실행");

setTimeout(() => {
    console.log("2. 매크로태스크 (setTimeout)");
}, 0);

Promise.resolve().then(() => {
    console.log("3. 마이크로태스크 (Promise)");
});

console.log("4. 동기 코드 종료");

/* 출력 순서:
1. 동기 코드 실행
4. 동기 코드 종료
3. 마이크로태스크 (Promise)
2. 매크로태스크 (setTimeout)
*/

Writing test.js


In [26]:
!node test.js

1. 동기 코드 실행
4. 동기 코드 종료
3. 마이크로태스크 (Promise)
2. 매크로태스크 (setTimeout)


In [27]:
%%writefile check.js

function checkStock(count) {
    return new Promise((resolve, reject) => {
        if (count > 0) {
            resolve("재고 있음");
        } else {
            reject("재고 없음");
        }
    });
}

checkStock(5)
    .then(res => console.log(res))
    .catch(err => console.log(err));

Overwriting check.js


In [28]:
!node check.js

재고 있음


In [29]:
%%writefile settled.js

const promiseTest = new Promise((resolve, reject) => {
    resolve("성공1");
    reject("실패");
    resolve("성공2");
});

promiseTest
    .then(res => console.log(res))
    .catch(err => console.log(err));

Writing settled.js


In [30]:
!node settled.js

성공1


In [31]:
%%writefile priority.js

setTimeout(() => console.log("A"), 0);

Promise.resolve().then(() => {
    console.log("B");
}).then(() => {
    console.log("C");
});

console.log("D");

Writing priority.js


In [32]:
!node priority.js

D
B
C
A


In [33]:
%%writefile catch.js

Promise.resolve("데이터1")
    .then(res => {
        console.log(res);
        throw new Error("중간 에러");
    })
    .catch(err => {
        console.log("에러 잡기:", err.message);
        return "복구 데이터";
    })
    .then(res => console.log("최종 결과:", res))
    .finally(() => console.log("종료"));

Writing catch.js


In [35]:
!node catch.js

데이터1
에러 잡기: 중간 에러
최종 결과: 복구 데이터
종료


In [36]:
%%writefile /content/ex1.js

// 1. 비동기 통신을 흉내내는 가상 함수 정의
function fetchUser(name) {
    return new Promise(resolve => {
        setTimeout(() => resolve({ id: 1, name: name }), 1000);
    });
}

function fetchPost(userId) {
    return new Promise(resolve => {
        setTimeout(() => resolve(`User ${userId}의 게시글입니다.`), 1000);
    });
}

// 2. 제너레이터 함수 정의
function* fetchGenerator() {
    console.log("데이터 로딩 시작...");
    const user = yield fetchUser("김철수"); // 일시 중단 및 Promise 반환
    console.log("사용자 정보 획득:", user);

    const post = yield fetchPost(user.id); // 재개 후 다시 중단
    console.log("게시글 정보 획득:", post);

    return post;
}

// 3. 내부 실행기(Runner) 구현
const gen = fetchGenerator();

// 첫 번째 next()는 첫 yield까지 실행하고 Promise를 반환함
gen.next().value.then(user => {
    // 두 번째 next(user)는 중단된 지점에 user 값을 넣어주며 재개함
    gen.next(user).value.then(post => {
        // 세 번째 next(post)는 마지막 결과를 반환함
        gen.next(post);
        console.log("모든 작업 완료");
    });
});

Writing /content/ex1.js


In [37]:
!node ex1.js

데이터 로딩 시작...
사용자 정보 획득: { id: 1, name: '김철수' }
게시글 정보 획득: User 1의 게시글입니다.
모든 작업 완료


In [38]:
%%writefile /content/ex1.js

async function getUserData(userId) {
    try {
        // 실제 존재하는 테스트용 API 주소로 변경
        const response = await fetch(`https://jsonplaceholder.typicode.com/users/${userId}`);

        if (!response.ok) throw new Error("데이터를 가져오지 못했습니다.");

        const data = await response.json();
        return data.name;
    } catch (error) {
        console.error("비동기 에러 발생:", error.message);
    }
}

// 실행을 위해 함수 호출 및 결과 출력
getUserData(1).then(name => {
    if (name) console.log("가져온 사용자 이름:", name);
});

Overwriting /content/ex1.js


In [39]:
!node ex1.js

가져온 사용자 이름: Leanne Graham


In [40]:
%%writefile fruit.js

/**
 * 간단한 해시 테이블 클래스 구현
 */
class HashTable {
  constructor(size = 10) {
    // 고정된 크기의 버킷 배열 생성
    this.buckets = new Array(size).fill(null).map(() => []);
    this.size = size;
  }

  /**
   * 해시 함수: 문자열 키를 숫자로 변환
   * @param {string} key
   */
  _hash(key) {
    let hash = 0;
    for (let i = 0; i < key.length; i++) {
      hash += key.charCodeAt(i);
    }
    return hash % this.size; // 버킷 크기로 나눈 나머지를 인덱스로 사용
  }

  /**
   * 데이터 저장 (Set)
   */
  set(key, value) {
    const index = this._hash(key);
    const bucket = this.buckets[index];

    // 이미 키가 존재하는지 확인 (충돌 대응 및 업데이트)
    const existingElement = bucket.find(element => element[0] === key);

    if (existingElement) {
      existingElement[1] = value; // 값 업데이트
    } else {
      bucket.push([key, value]); // 새로운 키-값 쌍 추가 (Chaining)
      console.log(`[저장] 인덱스 ${index}: ${key} -> ${value}`);
    }
  }

  /**
   * 데이터 조회 (Get)
   */
  get(key) {
    const index = this._hash(key);
    const bucket = this.buckets[index];

    const element = bucket.find(element => element[0] === key);
    return element ? element[1] : undefined;
  }

  /**
   * 전체 구조 출력
   */
  display() {
    console.log("\n--- 해시 테이블 내부 구조 ---");
    this.buckets.forEach((bucket, idx) => {
      const items = bucket.map(([k, v]) => `[${k}: ${v}]`).join(" -> ");
      console.log(`Bucket ${idx}: ${items || "비어 있음"}`);
    });
  }
}

/**
 * 실행 및 호출부
 */
const myTable = new HashTable(5);

// 1. 데이터 입력
myTable.set("Apple", "Red");
myTable.set("Banana", "Yellow");
myTable.set("Grape", "Purple");

// 2. 충돌 발생 유도 (키의 아스키 코드 합이 같은 경우나 해시 함수 특성상 발생 가능)
myTable.set("Melon", "Green");

// 3. 데이터 조회
console.log("\n조회 결과 (Apple):", myTable.get("Apple"));

// 4. 구조 확인
myTable.display();

Writing fruit.js


In [41]:
!node fruit.js

[저장] 인덱스 3: Apple -> Red
[저장] 인덱스 2: Banana -> Yellow
[저장] 인덱스 0: Grape -> Purple
[저장] 인덱스 2: Melon -> Green

조회 결과 (Apple): Red

--- 해시 테이블 내부 구조 ---
Bucket 0: [Grape: Purple]
Bucket 1: 비어 있음
Bucket 2: [Banana: Yellow] -> [Melon: Green]
Bucket 3: [Apple: Red]
Bucket 4: 비어 있음


In [42]:
%%writefile counter.js

const userLogs = ["login", "view_page", "login", "logout", "view_page", "view_page"];

function countActivities(logs) {
  const hashCounter = {}; // 해시 테이블 역할을 하는 객체

  for (let activity of logs) {
    // 해시의 Key는 고유하므로, 해당 키가 있으면 +1, 없으면 1로 초기화
    hashCounter[activity] = (hashCounter[activity] || 0) + 1;
  }
  return hashCounter;
}

console.log(countActivities(userLogs));
// 결과: { login: 2, view_page: 3, logout: 1 }

Writing counter.js


In [43]:
!node counter.js

{ login: 2, view_page: 3, logout: 1 }
